# Experiment Evaluation

This programm uses source code from the file ```MSR_map_coil.ipynb``` created by *Chiara Weckmann*.
___
Created on 29. Apr. 2026 by Gregor Bock 

(0378 1735; ge27doc)

## Imports

In [ ]:
import numpy as np                                          # general numerical calculations
import matplotlib.pyplot as plt                             # Plotting
import matplotlib.tri as mtri                               # Plotting triangular mesh (only if plotting of mesh is uncommented)
from mpl_toolkits.axes_grid1 import make_axes_locatable     # To move colourbar (collision with axis label)
import bfieldtools
import os

# Self made functions and classes
import External_functions as fkt                            # longer plots, file readouts and other larger functions are stored in this file to save space
from External_Classes import Coil_Layup, Mu_material        # Properties and methods of the Coil Layup and mu_metal are stored here (e.g. meshes, B-field by Biot-Savart, Stream_function calculation)

## Load Experiment Data

The data is loaded from a *map* directory, which has a *points* directory in which there are ```.npz``` (numpy-zip) files which hold the data for every point. The shape of the map is dertermined by the values in the ```.npz```-files as well as the shift given in this block.

In [ ]:
# Specify folder path for .npz and points folder
folder_path_points_no_current = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\Experiments\Background_Offsets_2026-08-13_14-23-43\map\points"
folder_path_points_with_current = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\Experiments\Strom_30_yes_degauss_kurz_2026-08-13_16-48-56\map\points"

# If map was not centered correctly, the shift of the map in each direction can be adjusted here. (The centre of the room shall be at the origin)
shift_x = 0                 # shift of the mapped volume in x-direction
shift_y = 0                 # shift of the mapped volume in y-direction
shift_z = 0                 # shift of the mapped volume in z-direction

# Load data using the function "load_data_from_folder" from the "Ausgelagerte_Funktionen_Versuchsauswertung.py" file.
target_point_coord_exp, B_target_point_no_current, B_target_point_with_current = fkt.load_data_from_folder(folder_path_points_no_current, folder_path_points_with_current, shift_x, shift_y, shift_z)

## Plot the extracted Data

Plot each $B$-field component as well as the norm of the $B$-field of both maps (with and without coil current) directly next to each other for better comparison.

In [ ]:
fkt.plot_Experiment_results_coils(target_point_coord_exp, B_target_point_no_current, B_target_point_with_current)   # Ploting function -> see Ausgelagerte_Funktionen_Versuchsauswertung.py

## Data definition

In this block, all experiment data which is not extracted from the ```.npz``` -file is defined.
- MSR dimensions (including dimensions of the door and inner wood-faces)
- Parameters for coil layup
- Parameters for mesh generation

**generally only SI-base units!**

In [ ]:
# Inner shield (Data from CAD model "TUM - innen.step" located in P:\MSR\CAD\STEP)
shield_height = 2.341       # height of the inner most shield (2.341)
shield_width = 2.448        # width and depth of the inner most mu-metal layer (2.448)
shield_thickness = 2e-3     # thickness of the shield walls (2e-3)

empirical_corr_fac_shield = 1.00            # Empirical correction factor accounting for multiple imperfect mu-metal layers (decreases B)
shield_height *= empirical_corr_fac_shield
shield_width *= empirical_corr_fac_shield
empirical_correction_factor_coils = 1.00    # Empirical correction factor (decreases curvature)

# Door (Data from CAD model "TUM - innen.step" located in P:\MSR\CAD\STEP)
door_removal = True         # Boolean which states, if the door shall be seperated in the mesh or not
door_width = 0.95           # (width of the door) (0.95)
door_height = 2.004         # (height of the door) (2.004)
door_offset_x = 0.2375      # (distance from door frame to right wall) (0.2375)
door_floor_offset = 0.0     # (distance from door frame to floor) (0.0)

# Inner wood structure (Data from CAD model "TUM - innen.step" located in P:\MSR\CAD\STEP)
height_inner_wood = 2.211   # height of the inner wooden structure (2.211)
width_inner_wood = 2.356    # width of the inner wooden structure (2.356)
depth_inner_wood = 2.356    # depth of the inner wooden structure (2.356)

# Coil Layout
coil_diameter = 0.70                # diameter of the coils
z_dist = 0.700                      # vertical distance between coils
x_dist = 2.340                      # horizontal distance between coils (x-direction)
y_dist = 0.700                      # horizontal distance between coils (y-direction)
coil_plane_dist_to_origin_x = width_inner_wood/2 + 0.005 - (0.032)
coil_plane_dist_to_origin_x *= empirical_correction_factor_coils
coil_plane_dist_to_origin_y = depth_inner_wood/2  # distance from the center of the room to the plane where the coils are placed in y-direction
coil_plane_dist_to_origin_y *= empirical_correction_factor_coils
coil_plane_dist_to_origin_z = height_inner_wood/2 # distance from the center of the room to the plane where the coils are placed in z-direction
coil_plane_dist_to_origin_z *= empirical_correction_factor_coils
usable_length_x = width_inner_wood - 0.1          # usable length in x-direction for coil placement
usable_length_y = depth_inner_wood - 0.1          # usable length in y-direction for coil placement
usable_length_z = height_inner_wood - 0.1         # usable length in z-direction for coil placement
n_windings = 10                     # number of windings per coil
current = 0.00003                   # current in A flowing through the coils

# Note: One can give a scalar input for n_windings or current, which will be applied to every coil
#       or a list of integer numbers, which need to has the same length as coils present in the layout (if the shapes do not match, an error accours!)

# Note: If you only want to have one coil in one direction, set acourding distance to zero and useable_length slightly larger than diameter
# Note: If you want NO coil in one direction, do not set useable_length_{x,y,z} to zero but set {x,y,z}_dist to useable_length_{x,y,z}

# Mesh parameters (chosen due to computation performance):
num_calc_target_points_fine = 19        # number of target points of the magnetic field in each direction (should be an odd number) (25)
num_calc_target_points_coarse = 5       # number of target points of the magnetic field in each direction (should be an odd number) (5)
safety_distance = -0.50                 # Distance between coil plane and closest target point (needed to avoid errors from the assumption of infinite permeability) (0.05)
refined_grid_quality_coilplane = 20     # number of initial nodes in one direction of the coil-plane (refined mesh) (25)
coursened_grid_quality_coilplane = 8    # number of initial nodes in one direction of the coil-plane (coarsened mesh) (12)
grid_quality_mu_metal = 8               # number of initial nodes in one direction of the mu_material plane (12)

## Call instance to Coil_Layup class

The class ```Coil_Layup``` from the file **Externe_Classes.py** is called to give the coil geometries by inputting all the geometric data.

Furthermore, this block plots the coils which were specified by the geomertic data as well as the centers of these coils.

In [ ]:
Coils = Coil_Layup(coil_diameter, x_dist, y_dist, z_dist, n_windings, current, coil_plane_dist_to_origin_x, coil_plane_dist_to_origin_y, coil_plane_dist_to_origin_z, usable_length_x, usable_length_y, usable_length_z)
all_circles = Coils.all_coils  # Shape: (n_coils_xy, 100, 3)

# Plot
# Plot the coil-wires
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
for k in range(all_circles.shape[0]):
    ax.plot(all_circles[k, :, 0], all_circles[k, :, 1], all_circles[k, :, 2], color='blue')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')

# Plot the centre of each coil
fig_grid = plt.figure()
ax_grid = fig_grid.add_subplot(111, projection='3d')
ax_grid.scatter(Coils.grid[:, :, 0], Coils.grid[:, :, 1], Coils.grid[:, :, 2], color='red', s=100)
ax_grid.set_xlabel('x')
ax_grid.set_ylabel('y')
ax_grid.set_zlabel('z')

plt.show()

## $B$-field by Biot-Savart

Calculate the theoretical $B$-field produced by the coils using the law of Biot-Savart.
\begin{equation}
    \mathbf{B}(\mathbf{r}) = \frac{\mu_0}{4\pi} \int \frac{I \, \text{d}\ell \times (\mathbf{r} - \mathbf{r}')}{|\mathbf{r} - \mathbf{r}'|^3} \Rightarrow \text{d}{\bf{B}} = \frac{{\mu _0 }}{{4\pi }}\frac{{I \text{d}\ell \times {\bf{\hat R}}}}{{R^2 }}
\end{equation}

In [ ]:
target_point_coord_calc_fine = fkt.calculation_target_points(num_calc_target_points_fine, coil_plane_dist_to_origin_x, coil_plane_dist_to_origin_y, coil_plane_dist_to_origin_z, safety_distance)

B_coil_predicted_fine = Coils.coil_field_by_Biot_Savart(target_point_coord_calc_fine)       # Calculate B-field for a high resolution -> sanity check

# Plot
# Plot the magnetic field due to Biot-Savart law
Bmag = np.linalg.norm(B_coil_predicted_fine, axis=1)                                        # Calculate norm of $B$

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(
    target_point_coord_calc_fine[:, 0],
    target_point_coord_calc_fine[:, 1],
    target_point_coord_calc_fine[:, 2],
    alpha = 0.6,
    c=Bmag,
    s=30,
    cmap='viridis'
)
ax.set_title('Biot-Savart')
fig.colorbar(sc, ax=ax, label=r'$|\mathbf{B}_{coil}|$ [T]')

## Coursen target point grid

This highly resolved grid can not be used for further computations (too computationally expensive) and can also not be used to make valid comparisons with measurements, since all experimental map-resolutions will be significantly lower! Therefore the target point grid is coursened and the computed $B$-field (only coils) is expressed on the coarsened grid.

In [ ]:
# Compute the target point coordinates of the coarse grid which is similar to the one typically measured in the MSR (here 5x5x5, MSR Experiments 5x5x3)
Max = np.max(target_point_coord_exp)
target_point_coord_calc_coarse = fkt.calculation_target_points(num_calc_target_points_coarse, Max, Max, Max, safety_distance=0)

B_coil_predicted_coarse = Coils.coil_field_by_Biot_Savart(target_point_coord_calc_coarse)   # Calculate B-field for a low resolution -> computationally achievable and comparable to experiment

# Plot
# Plot the magnetic field due to Biot-Savart law
Bmag = np.linalg.norm(B_coil_predicted_coarse, axis=1)                                      # Calculate norm of $B$

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(
    target_point_coord_calc_coarse[:, 0],
    target_point_coord_calc_coarse[:, 1],
    target_point_coord_calc_coarse[:, 2],
    alpha = 0.7,
    c=Bmag,
    s=100,
    cmap='viridis'
)
fig.colorbar(sc, ax=ax, label=r'$|\mathbf{B}_{coil}|$')

## Derive stream function from surface current

Here I compute the streamfunction directly from the surface current. The defining equation is

\begin{equation}
    J(r) = \nabla_\parallel \psi(r) \times n
\end{equation}

Workflow:
- Create a mesh (highly resolved -> later coursening)
- Create arrays which hold the indices of those faces, which are inside, outside or directly on the coils.
- Depending on the array each face index is in, either set the streamfunction there to
    - 0 if the face is inside the coil
    - $I_{tot} \cdot \Delta x$ if the face is inside the coil
    - $\frac{1}{2} I_{tot} \cdot \Delta x$ if the coil intersects the face
- Since we will later need the streamfunction at each vertex, we compute the streamfunction there by averaging over all adjacent faces.

Some notes:
- It is much faster and robust than the computation via the inverse formulation of the coupling matrix and B-field
- It may have problems with very small coils as well as very small distances between cells, since the coil thickness gets very large (not representing the experiment in the best manner)
- The modeling approaches the real experiment for very small faces -> highly resolved mesh (line thickness of the coil is at least two vertices but not more than three)

Improvements:
- The method ```.stream_function_coils``` takes only scalar input values for ```current``` and ```n_windings```. This will be improved, but has low priority at the moment.

In [ ]:
Coils.create_mesh(refined_grid_quality_coilplane, door_removal, door_width, door_height, door_floor_offset, door_offset_x)      # Create a triangular mesh on the plane where the coils are located

# # Plot the mesh in 2D
# x = Coils.mesh_back.vertices[:, 0]
# z = Coils.mesh_back.vertices[:, 2]
# faces = Coils.mesh_back.faces

# triang = mtri.Triangulation(x, z, faces)
# fig, ax = plt.subplots(figsize=(6, 6))
# ax.triplot(triang, color='k', linewidth=0.8)
# ax.set_aspect('equal')
# ax.set_title('XZ wall with door')
# plt.show()

# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# inner_idx=bfieldtools.utils.find_mesh_boundaries(Coils.total_planes)
# inner_vertex_idx=inner_idx[0]
# for idx in inner_vertex_idx:
#     boundary=Coils.total_planes.vertices[idx]
#     ax.scatter(boundary[:,0],boundary[:,1],boundary[:,2])

# fig = plt.figure(figsize=(8,6))
# ax = fig.add_subplot(111, projection='3d')
# #fig.subplots_adjust(left=0.0, right=30.0, bottom=0.0, top=1.0)
# verts=Coils.total_planes.vertices
# ax.scatter(verts[:,0],verts[:,1],verts[:,2],alpha=0.1,label='coil planes')
# ax.scatter(target_point_coord_calc_fine[:,0],target_point_coord_calc_fine[:,1],target_point_coord_calc_fine[:,2],label='MSR Map Points')
# # Axis labels
# ax.set_xlabel('x [m]')
# ax.set_ylabel('y [m]')
# ax.set_zlabel('z [m]')
# plt.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
stream_func_coil_refined = Coils.stream_function_coils_verts(current, n_windings)         # Note: At the moment this function only accepts scalar values for current and n_windings!!! Improvements in progress but low priority
# len_st_func = int(len(stream_func_coil_refined)/6)
# stream_func_coil_refined[-len_st_func:] = -stream_func_coil_refined[-len_st_func:]
fkt.plot_stream_function(Coils.total_planes.vertices, stream_func_coil_refined)     # Plot stream_function both in isometric 3D plot and each side

## Shield meshing class

To account for the $\mu$-metal a meshed geometry accourding to the specified geometric data can be created by calling the class ```Mu_material``` from the file **Externe_Classes.py**. This will output a triangular mesh of the $\mu$-metal surface.

This block also plots the mesh (of one planar surface), the mesh boundaries (should be an empty plot) as well as the cells in 3D just as in the original programm.

In [ ]:
mu_grid = Mu_material(grid_quality_mu_metal, shield_height, shield_width, shield_thickness)          # Create a triangular mesh at the plane where the inner shell of mu-metal is located

# # Plot
# # plot meshboundaries of the mu-metal
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# inner_vertex_idx_shield = mu_grid.inner_idx[0]
# for idx in inner_vertex_idx_shield:
#     boundary=mu_grid.total_shield.vertices[idx]
#     ax.scatter(boundary[:,0],boundary[:,1],boundary[:,2])

# # plot vertices
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# verts_shield = mu_grid.total_shield.vertices
# ax.scatter(verts_shield[:,0],verts_shield[:,1],verts_shield[:,2],alpha=0.5)

# # plot vertices of the shield and points inside the shield
# points_inside = mu_grid.total_shield.vertices - mu_grid.thickness * mu_grid.total_shield.vertex_normals
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(verts_shield[:,0],verts_shield[:,1],verts_shield[:,2],alpha=0.3)
# ax.scatter(points_inside[:,0],points_inside[:,1],points_inside[:,2],alpha=0.3)

## Mesh the coil plane and create streamfunction from current

Now a coarse mesh needs to be made. The mesh we already created has too high resulution and is computationally too expensive to use for the calculation.

Create streamfunction on the coarsened mesh just as before, assigning values based on the number of coils enclosing the cell.

In [ ]:
total_planes = Coils.create_mesh(coursened_grid_quality_coilplane, door_removal, door_width, door_height, door_floor_offset, door_offset_x)
stream_func_coil_coarse = Coils.stream_function_coils_verts(current, n_windings)      # Note: At the moment this function only accepts scalar values for current and n_windings!!! Improvements in progress but low priority

# # Plot the mesh in 2D
# x = Coils.mesh_back.vertices[:, 0]
# z = Coils.mesh_back.vertices[:, 2]
# faces = Coils.mesh_back.faces

# triang = mtri.Triangulation(x, z, faces)
# fig, ax = plt.subplots(figsize=(6, 6))
# ax.triplot(triang, color='k', linewidth=0.8)
# ax.set_aspect('equal')
# ax.set_title('XZ wall with door')
# plt.show()

# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# inner_idx=bfieldtools.utils.find_mesh_boundaries(Coils.total_planes)
# inner_vertex_idx=inner_idx[0]
# for idx in inner_vertex_idx:
#     boundary=Coils.total_planes.vertices[idx]
#     ax.scatter(boundary[:,0],boundary[:,1],boundary[:,2])

# fig = plt.figure(figsize=(8,6))
# ax = fig.add_subplot(111, projection='3d')
# #fig.subplots_adjust(left=0.0, right=30.0, bottom=0.0, top=1.0)
# verts=Coils.total_planes.vertices
# ax.scatter(verts[:,0],verts[:,1],verts[:,2],alpha=0.1,label='coil planes')
# ax.scatter(target_point_coord_calc_coarse[:,0],target_point_coord_calc_coarse[:,1],target_point_coord_calc_coarse[:,2],label='MSR Map Points')
# # Axis labels
# ax.set_xlabel('x [m]')
# ax.set_ylabel('y [m]')
# ax.set_zlabel('z [m]')
# plt.legend()
# plt.tight_layout()
# plt.show()

# fkt.plot_stream_function(Coils.total_planes.vertices, stream_func_coil_coarse)

## Finding the Coupling-matrix $C_{ij}$

**Prerequisites**:

In the calculatetion we are using the stream function $\psi(r)$ which can be closely linked to the surface current with
\begin{gather}
    {j}({r}) = \nabla_\parallel \psi(r) \times n(r)\\
    \psi(r) - \psi(r_0) = \int_{r_0}^r j(r') \cdot (\text{d}l \times n')
\end{gather}
where current_density $j$, normal-vector $n$ and position $r$ are vectors in 3D and the stream function $\psi$ is a scalar quantity.

In the programm we define the stream function as the sum of as weight $s_i$ times a hat-function $h_i$ at every vertex $i$. The hat function is a simple function which is 1 at the vertex $i$ and zero at all other verteces. Between the verteces it gets interpolated linearly!

\begin{equation}
    \psi(r) = \sum_{i=0}^{\text{number of vertices}} s_i \cdot h_i(r)
\end{equation}

Following this we will express all obperations with the stream function $\psi$ as functions of its weights $s_i$!

**Workflow of the program**:
- Calculate scalar potential matrices $U_{coil}$ and $U_{shield}$:
\begin{equation}
    \Phi(r_i) = \sum_{j=i}^{N_{vertices}} U_{ij, \text{coil}}s_{j, \text{coil}} = -\sum_{j=i}^{N_{vertices}} U_{ij, \text{shield}}s_{j, \text{shield}}
\end{equation}
where $\Phi(r_i)$ is the scalar potential, $s_j$ is the stream function at vertex $j$ and U_{ij} is the scalar potential matrix. (See paper: Magnetic field modeling with surface currents. Part II. Implementation and usage of bfieldtools; Equation (12))

- From scalar potential matrices $U_{ij}$ compute coil-shiel-coupling matrix $M_{ij}$
\begin{gather}
    U_{shield} \cdot M_{tot\_shield} = U_{coils} \\
    s_{\text{shield}} = - U_{\text{shield}}^{-1} \cdot U_{\text{coil}} \cdot s_{\text{coil}} = M_{tot\_shield} \cdot s_{\text{coil}}
\end{gather}

- Calculate the magnetic coupling matrix $C_{ij\alpha}$ of the $\mu$-metal in all three space directions
\begin{align}
    B_{\alpha,\text{shield}}(r_k) = \sum_{i=1}^{N_{vertices}} C_{ki\alpha,\text{shield}}s_{i,\text{shield}} && \alpha \in \{x,y,z\}
\end{align}
where $B_{\alpha,\text{shield}}(r_k)$ is the magnetic field at $r_k$ in direction $\alpha$ produced by the shield, and $s_{i,\text{shield}}$ is the streamfunction in the shield at vertex $i$.

- Calculate the magnetic coupling matrix $C^*_{ij\alpha,\text{shield}}$ of the $\mu$-metal as if it was caused by the coil-plane (change of reference)
 \begin{equation}
    C^*_{\alpha,\text{shield}} = C_{\alpha,\text{shield}} \cdot M_{tot\_shield}
 \end{equation}

- Calculate the magnetic coupling matrix $C^*_{ij\alpha,\text{coil}}$ of the coils
\begin{align}
    B_{\alpha,\text{coil}}(r_k) = \sum_{i=1}^{N_{vertices}} C_{ki\alpha,\text{coil}}s_{i,\text{coil}} && \alpha \in \{x,y,z\}
\end{align}
where all quantities are analog to those of the shield case.

- At the boundaries of the coil plane, it holds that in any case no current can flow over the edge, resulting in a streamfunction which is constant and can be set to zero (gauge). (**Note**, that since the shield of the MSR is connected at the edges, allowing current-flow over the edge. Therefore the edges of the shield are no boundaries! Only the coil-plane needs assigned boundary values. This is done in the class method ```Coils.stream_func_coils_verts()```)

- Optain the total coupling matrix $C_{ij\alpha, \text{total}}$ of both coils and shield by adding both together
\begin{equation}
    C_{ij\alpha, \text{total}} = C^*_{ij\alpha,\text{shield}} + C_{ij\alpha,\text{coil}}
\end{equation}

- Reshape the coil coupling matrix $C^*_{ij\alpha,\text{coil}}$ and the total coupling matrix $C_{ij\alpha, \text{total}}$ as well as the predicted $B$-field due to the coils alone to better use it later on (shape: N, 3, M -> 3N, M)



In [ ]:
# Scalar-Potential Coupling Matrix U_{ij} (only dependent on geometrical data of meshes)
    # mu-material
U_coupling_shield_inside = bfieldtools.mesh_magnetics.scalar_potential_coupling(mu_grid.total_shield, mu_grid.points_inside)
    # Coils
U_coupling_coils_inside = bfieldtools.mesh_magnetics.scalar_potential_coupling(Coils.total_planes, mu_grid.points_inside)

# total shield coupling which links the coil plane to the shield plane U_shield * M = U_coil
Coil_Shield_coupling = -np.linalg.solve(U_coupling_shield_inside, U_coupling_coils_inside)

# B-field coupling matrix C_shield of the shield alone (only dependent on gemetrical data of mu-mesh and target point coordinates)
Coupling_shield = bfieldtools.mesh_magnetics.magnetic_field_coupling(mu_grid.total_shield, target_point_coord_calc_fine)

# B-field coupling matrix C*_shield of the shield alone, seen as a secondary source from the coil plane perspective C*_shield = C_shield * M
secondary_Coupling_shield = np.tensordot(Coupling_shield, Coil_Shield_coupling, axes=([2],[0]))

# B-field coupling matrix C_coil of the coil plane (only dependent on gemetrical data of coil-mesh and target point coordinates)
Coupling_coil = bfieldtools.mesh_magnetics.magnetic_field_coupling(Coils.total_planes, r = target_point_coord_calc_fine)

In [ ]:
# Combined coupling matrix C_total which incorporates the effect of the shield as a secondary source and the coil plane as a primary source
total_Coupling = Coupling_coil + secondary_Coupling_shield #* shield1_fac + secondary_Coupling_shield2 * shield2_fac + secondary_Coupling_shield3 * shield3_fac + secondary_Coupling_shield4 * shield4_fac # Note: The minus here is directly from the equation in the paper (see markdown)

# Suppose C_coil has shape (N, 3, M) (N...number of target points, M...number of vertices in coil plane)
N, _, M = Coupling_coil.shape
Coupling_coil_flat = Coupling_coil.reshape(3 * N, M)

# Suppose C_tot has shape (N, 3, M) (N...number of target points, M...number of vertices in coil plane)
N, _, M = total_Coupling.shape
total_Coupling_flat = total_Coupling.reshape(3 * N, M)

# Suppose B_predicted has shape (N, 3) (N...number of target points)
N, _ = B_coil_predicted_fine.shape
B_coil_predicted_flat = B_coil_predicted_fine.reshape(3 * N)

# Debug
print(f'Shapes of the different matrices:\n'
      f'   U_coupling_shield_inside:  {U_coupling_shield_inside.shape}\n'
      f'   U_coupling_coils_inside:   {U_coupling_coils_inside.shape}\n'
      f'   Coil_Shield_coupling:      {Coil_Shield_coupling.shape}\n'
      f'   B_coupling_shield:         {Coupling_shield.shape}\n'
      f'   secondary_Coupling_shield: {secondary_Coupling_shield.shape}\n'
      f'   Coupling_coil:             {Coupling_coil.shape}\n'
      f'   total_Coupling:            {total_Coupling.shape}\n'
      f'   Coupling_coil_flat:        {Coupling_coil_flat.shape}\n'
      f'   total_Coupling_flat:       {total_Coupling_flat.shape}')

## Calculate $B$-field from coupling matrix and stream function

- In a first step, we find the stream function of the coils making use of
\begin{equation}
    B_{\text{coil}} = C_{\text{coil}} \cdot s_{\text{coil}}
\end{equation}

- Using the fact that
\begin{gather}
    B_{\text{tot}} = B_{\text{coil}} + B_{\text{shield}} = C_{\text{coil}} \cdot s_{\text{coil}} + C_{\text{shield}} \cdot s_{\text{shield}} \\
    = (C_{\text{coil}} - C_{\text{shield}}\cdot U_{\text{shield}}^{-1}\cdot U_{\text{coil}}) s_{\text{coil}} \\
    = C_{tot} s_{\text{coil}}
\end{gather}
the total magnetic field can directly be computed!

In [ ]:
# Some corrections to change current flow direction
len_st_func = int(len(stream_func_coil_coarse)/6)
stream_func_coil_coarse[-len_st_func:] = -stream_func_coil_coarse[-len_st_func:]

# Use coil stream function and total coupling matrix to compute total magnetic field
B_tot_flat = total_Coupling_flat @ stream_func_coil_coarse

# Extract values in B_tot_flat of shape (3*number_target_points) into B_tot of shape (number_target_points, 3)
N, _ = B_coil_predicted_fine.shape      # N = number_of_target_points
B_tot = B_tot_flat.reshape(N, 3)        # B_tot has shape (number_target_points, 3)

# Debug
print(f'Shapes of the different matrices:\n'
      f'   Stream function of coils: {stream_func_coil_coarse.shape}\n'
      f'   B_tot_flat:               {B_tot_flat.shape}\n'
      f'   B_tot:                    {B_tot.shape}\n'
      f'   Target points coord fine: {target_point_coord_calc_fine.shape}')

# Plots
fig_B_tot = plt.figure()
ax_B_tot = fig_B_tot.add_subplot(111, projection='3d')
sc_B_tot = ax_B_tot.scatter(
    target_point_coord_calc_fine[:, 0],
    target_point_coord_calc_fine[:, 1],
    target_point_coord_calc_fine[:, 2],
    c=np.linalg.norm(B_tot, axis=1),
    s=100,
    cmap='viridis',
    alpha = 0.1,
)
fig_B_tot.colorbar(sc_B_tot, ax=ax_B_tot, label=r'$|\mathbf{B}_{\text{total}}|$')

# Save to .npz with named keys
file_dir = os.path.dirname(folder_path_points_with_current)
parent_dir = os.path.dirname(file_dir)
filename = 'B_tot_results\\B_tot_result_' + os.path.basename(parent_dir) + '.npz'

np.savez(filename, B_field = B_tot, Biot_Savart = B_coil_predicted_fine, points = target_point_coord_calc_fine, stream_function = stream_func_coil_coarse, Coupling_matrix = total_Coupling, d_coil = coil_diameter, current = current, n_windings = n_windings, all_coils = all_circles)

# Evaluation

Make nice plots and Sanity checks here!

## total streamfunction on coil plane

Compute the equivalent total streamfunction on the coil_plane. This is done to make sanity chacks later on but is also very memory intensive, since the streamfunction is calculated via the inverse of the problem:

\begin{equation}
    B_{i, \alpha} = C_{i, j, \alpha} \cdot \psi_j
\end{equation}

In [ ]:
if False: # Only run if of interest (This block takes a while)
    # For the sake of completeness, the equivalent total stream function is computed on the coil-planes
    stream_func_tot, _, _, _ = np.linalg.lstsq(total_Coupling_flat, B_tot_flat, rcond=None)

    # Debug
    print(f'The shape of the total streamfunction is: {stream_func_tot.shape}')

    fkt.plot_stream_function(Coils.total_planes.vertices, stream_func_tot)

## Plot streamfunction of coil

Here, we want to derive the streamfunction from:
\begin{equation}
    \mathbf{B} = \mathbf{C} \cdot \mathbf{s}
\end{equation}
Sadly, $\mathbf{C}$ is in general not a square matrix and not invertable. Therefore we can solve the problem using the numpy least square algorythm which searches for the solution $s_{LQ}$ which minimises:
\begin{equation}
    \min_{s}||Cs_{LQ} - B||_2^2
\end{equation}

However, this may lead to an overfitting of the actual streamfunction. In an effort to decrese the error which is generated by our solution, we accept fast and unphysical changes in the streamfunction. To solve this issue, we introduce a *Tikhonov* (Laplace) regularisation. This regularisation uses the *Laplace Operator* $\text{L}$, which can be seen as second derivative of our scalar field. It is added in the cost function to penalise fast changes in the gradient of the streamfunction and thereby reduces overfitting. The improved cost-function looks like
\begin{equation}
    \min_{s}||Cs_{LQ} - B||_2^2 + \alpha||\text{L} s||_2^2
\end{equation}
where $\alpha$ is a constant weighing "smoothness" against accuracy.

This can be reformulated in a single optimization problem
\begin{equation}
    \min_{s}||\begin{bmatrix}C\\ \sqrt{\alpha}\text{L}\end{bmatrix}s-\begin{bmatrix}B\\0\end{bmatrix}||_2^2
\end{equation}

The problem arises, how to implement the Laplace Operator $\text{L}$ in the discrete code. Here the discretised Laplace–Beltrami operator is used.

In [ ]:
if False: # Only run if of interest (This block takes a while)
    alpha = 1e-3  # tune as needed
    stream_func_coil_test, L = Coils.stream_function_coils_regularized(
        Coupling_coil_flat,
        B_coil_predicted_flat,
        alpha=alpha,
    )
    fkt.plot_stream_function(Coils.total_planes.vertices, stream_func_coil_test)



## Strom

Berechne den Strom aus der Streamfunction um nachzuprüfen, ob der Output sinnvoll ist.

Außerdem: Darüber nachdenken, ob ich das Problem rückwerts angehen kann, um direkt ein Ergebnis zu finden, und nicht über das B-Feld zu müssen.

Allgemein gilt:
\begin{gather}
    j(r) = \nabla_\parallel \psi(r) \times n(r) \\
    \psi(r)-\psi(r_0) = \int_{r_0}^r j(r')\cdot (dl' \times n')
\end{gather}

In [ ]:
stream_func_coil_coarse = np.asarray(stream_func_coil_coarse, dtype=float)

plane_meshes = [
    ("Top", Coils.mesh_top),
    ("Bottom", Coils.mesh_bottom),
    ("Back", Coils.mesh_back),
    ("Front", Coils.mesh_front),
    ("right", Coils.mesh_right),
    ("left", Coils.mesh_left),
]

# Match each sub-mesh vertex to the corresponding index in the combined mesh.
combined_vertices = Coils.total_planes.vertices
rounded_combined = np.round(combined_vertices, decimals=12)

coord_to_index = {}
for idx, coord in enumerate(rounded_combined):
    key = tuple(coord)
    if key not in coord_to_index:
        coord_to_index[key] = idx


def get_plane_values(mesh, stream_values):
    if len(mesh.vertices) == 0:
        return np.array([]), np.array([]), np.array([]), np.array([])

    rounded_mesh = np.round(mesh.vertices, decimals=12)
    indices = []
    for coord in rounded_mesh:
        key = tuple(coord)
        if key not in coord_to_index:
            raise ValueError("A plane vertex could not be matched to the combined mesh vertices.")
        indices.append(coord_to_index[key])

    indices = np.array(indices, dtype=int)
    return mesh.vertices[:, 0], mesh.vertices[:, 1], mesh.vertices[:, 2], stream_values[indices]

fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=False, dpi=200)
axes = axes.ravel()

# Use one shared normalization for all panels so the colorbar corresponds to the same values.
vmin = float(np.min(stream_func_coil_coarse))
vmax = float(np.max(stream_func_coil_coarse))

for face_idx, ax in enumerate(axes):
    face_name, mesh = plane_meshes[face_idx]
    x_face, y_face, z_face, s_face = get_plane_values(mesh, stream_func_coil_coarse)

    xr = np.ptp(x_face)
    yr = np.ptp(y_face)
    zr = np.ptp(z_face)

    ranges = {'x': xr, 'y': yr, 'z': zr}
    const_axis = min(ranges, key=ranges.get)

    if const_axis == 'x':
        a, b = y_face, z_face
        a_name, b_name = '$y$ [m]', '$z$ [m]'
    elif const_axis == 'y':
        a, b = x_face, z_face
        a_name, b_name = '$x$ [m]', '$z$ [m]'
    else:
        a, b = x_face, y_face
        a_name, b_name = '$x$ [m]', '$y$ [m]'

    sort_indices = np.lexsort((b, a))

    a_sorted = a[sort_indices]
    b_sorted = b[sort_indices]
    s_sorted = s_face[sort_indices]

    # Scatter plot of the streamfunction values (unchanged)
    sc = ax.scatter(
        a_sorted,
        b_sorted,
        c=s_sorted,
        cmap='RdBu_r',
        vmin=vmin,
        vmax=vmax,
        s=50,
    )

    # Quiver plot of the current direction as the gradient of the streamfunction,
    # rotated by 90 degrees in the plotted plane.
    if len(a_sorted) > 1:
        # Estimate a local gradient at every point from its nearest neighbours.
        qx = np.zeros_like(a_sorted, dtype=float)
        qy = np.zeros_like(b_sorted, dtype=float)
        n_neighbors = min(6, max(3, len(a_sorted) // 4))

        for i in range(len(a_sorted)):
            dist = np.sqrt((a_sorted - a_sorted[i])**2 + (b_sorted - b_sorted[i])**2)
            neighbor_idx = np.argsort(dist)[1:1 + n_neighbors]
            if len(neighbor_idx) < 2:
                continue

            local_a = a_sorted[neighbor_idx] - a_sorted[i]
            local_b = b_sorted[neighbor_idx] - b_sorted[i]
            local_s = s_sorted[neighbor_idx]

            A = np.column_stack((np.ones(len(neighbor_idx)), local_a, local_b))
            coeffs, _, _, _ = np.linalg.lstsq(A, local_s, rcond=None)
            da_local = coeffs[1]
            db_local = coeffs[2]

            # Rotate by 90 degrees to obtain the current direction.
            qx[i] = -db_local
            qy[i] = da_local

        # Normalize the arrow vectors to a visible length.
        qmag = np.sqrt(qx**2 + qy**2)
        max_mag = float(np.max(qmag)) if qmag.size > 0 else 0.0
        if max_mag > 0:
            qx = qx / max_mag
            qy = qy / max_mag
        else:
            qx = np.zeros_like(qx)
            qy = np.zeros_like(qy)

        face_scale = float(np.maximum(0.03 * np.maximum(np.ptp(a_sorted), np.ptp(b_sorted)), 0.003))
        qx = qx * face_scale
        qy = qy * face_scale

        ax.quiver(
            a_sorted,
            b_sorted,
            qx,
            qy,
            angles='xy',
            scale_units='xy',
            scale=0.5,
        )

    ax.set_xlabel(a_name)
    ax.set_ylabel(b_name)
    ax.set_title(face_name)
    ax.axis('equal')

divider = make_axes_locatable(fig.axes[-1])
cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(sc, cax=cax, label='Streamfunction $\\Psi$')
# plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()